# Phase 2 & 3: Clean Alignment, Extraction and Split Setup
This notebook loads the pre-extracted windowed features, performs inner-joins on composite key windows, and splits subjects into train/val/test groups.

In [ ]:
print("[BACKGROUND] Importing environment modules...")
import os
import sys
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath('../'))
from src.data.split import SubjectSplitter
print("[STATUS] Environment successfully initialized.")

In [ ]:
print("[BACKGROUND] Searching for pre-extracted datasets...")
# Drive vs local check
search_paths = [
    '../../dataset_extracted/',
    '../../certified_data/',
    '../data/raw/'
]
extracted_dir = None
for p in search_paths:
    if os.path.exists(p) and any(f.endswith('.csv') for f in os.listdir(p)):
        extracted_dir = p
        break

if extracted_dir is None:
    print("[WARNING] Pre-extracted feature CSV files not found. Creating mock data for dry-run...")
    os.makedirs('../data/raw/', exist_ok=True)
    # Mock feature tables
    df_face = pd.DataFrame({
        'subject_id': ['s01']*5 + ['s02']*5 + ['s03']*5 + ['s04']*5 + ['s05']*5,
        'task_id': ['Stroop']*25,
        'window_index': list(range(5))*5,
        'label': [1]*10 + [0]*15,
        **{f'face_{i}': np.random.randn(25) for i in range(18)}
    })
    df_voice = pd.DataFrame({
        'subject_id': ['s01']*5 + ['s02']*5 + ['s03']*5 + ['s04']*5 + ['s05']*5,
        'task_id': ['stroop']*25,
        'window_index': list(range(5))*5,
        **{f'voice_{i}': np.random.randn(25) for i in range(12)}
    })
    df_physio = pd.DataFrame({
        'subject_id': ['S01']*5 + ['S02']*5 + ['S03']*5 + ['S04']*5 + ['S05']*5,
        'task_id': ['STROOP']*25,
        'window_index': list(range(5))*5,
        **{f'physio_{i}': np.random.randn(25) for i in range(5)}
    })
    df_face.to_csv('../data/raw/face_certified.csv', index=False)
    df_voice.to_csv('../data/raw/voice_certified.csv', index=False)
    df_physio.to_csv('../data/raw/physio_certified.csv', index=False)
    extracted_dir = '../data/raw/'

print(f"[INFO] Loading pre-extracted features from directory: {os.path.abspath(extracted_dir)}")

# Load files by matching patterns
files = os.listdir(extracted_dir)
f_csv = [f for f in files if 'face' in f.lower() and f.endswith('.csv')][0]
v_csv = [f for f in files if 'voice' in f.lower() and f.endswith('.csv')][0]
p_csv = [f for f in files if 'physio' in f.lower() and f.endswith('.csv')][0]

df_face = pd.read_csv(os.path.join(extracted_dir, f_csv))
df_voice = pd.read_csv(os.path.join(extracted_dir, v_csv))
df_physio = pd.read_csv(os.path.join(extracted_dir, p_csv))
print(f"[STATUS] Datasets loaded. Face Shape: {df_face.shape}, Voice Shape: {df_voice.shape}, Physio Shape: {df_physio.shape}")

In [ ]:
print("[BACKGROUND] Normalizing task IDs and keys structure...")
for df in [df_face, df_voice, df_physio]:
    df['subject_id'] = df['subject_id'].astype(str).str.lower().str.strip()
    df['task_id'] = df['task_id'].astype(str).str.lower().str.strip()
    df['window_index'] = df['window_index'].astype(int)
    df['sync_key'] = df['subject_id'] + "_" + df['task_id'] + "_" + df['window_index'].astype(str)

print("[BACKGROUND] Aligning modalities via left-joining using physio as base...")
# Drop metadata columns from other dataframes to keep merged table clean
df_face_clean = df_face.drop(columns=['subject_id', 'task_id', 'video_id', 'window_index', 'window_start', 'window_end', 'label'], errors='ignore')
df_voice_clean = df_voice.drop(columns=['subject_id', 'task_id', 'video_id', 'window_index', 'window_start', 'window_end', 'label'], errors='ignore')

synced_df = df_physio.merge(df_face_clean, on='sync_key', how='left')
synced_df = synced_df.merge(df_voice_clean, on='sync_key', how='left')

print(f"[STATUS] Modality alignment finished. Synced samples remaining: {synced_df.shape[0]}")

In [ ]:
print("[BACKGROUND] Performing Subject-Level Baseline Normalization...")
# Define all feature columns
face_cands = ['left_ear', 'right_ear', 'avg_ear', 'blink_velocity', 'brow_descent_left', 'brow_descent_right', 'brow_asymmetry', 'lip_compression', 'jaw_tension', 'mouth_corner_pull', 'forehead_tension', 'face_height_norm', 'head_tilt', 'temporal_x_var', 'temporal_y_var', 'eye_openness_ratio', 'landmark_confidence', 'nose_wrinkle']
voice_cands = ['f0_mean', 'f0_std', 'f0_range', 'jitter_percent', 'shimmer_db', 'hnr', 'speaking_rate_proxy', 'voice_intensity', 'high_freq_ratio', 'spectral_flux', 'pause_ratio', 'voiced_fraction']
physio_cands = ['ecg_rate_mean', 'ecg_hrv_rmssd', 'ecg_hrv_sdnn', 'eda_scl_mean', 'resp_rate_mean']

face_cols = [c for c in face_cands if c in synced_df.columns]
voice_cols = [c for c in voice_cands if c in synced_df.columns]
physio_cols = [c for c in physio_cands if c in synced_df.columns]
feature_cols = face_cols + voice_cols + physio_cols

# Calculate resting baseline averages per subject during 'baseline' task
baseline_df = synced_df[synced_df['task_id'] == 'baseline']
subject_baselines = baseline_df.groupby('subject_id')[feature_cols].mean()
# Overall subject mean as fallback if 'baseline' task is missing
subject_overall_means = synced_df.groupby('subject_id')[feature_cols].mean()

normalized_rows = []
for sub in synced_df['subject_id'].unique():
    sub_df = synced_df[synced_df['subject_id'] == sub].copy()
    # Get baseline vector for this subject
    if sub in subject_baselines.index and not subject_baselines.loc[sub].isna().any():
        sub_base = subject_baselines.loc[sub]
    else:
        sub_base = subject_overall_means.loc[sub]
        
    # Subtract baseline vector from all tasks
    sub_df[feature_cols] = sub_df[feature_cols] - sub_base
    normalized_rows.append(sub_df)
    
synced_df = pd.concat(normalized_rows, ignore_index=True)
print("[STATUS] Baseline normalization complete. Features converted to relative change indices.")

In [ ]:
print("[BACKGROUND] Executing subject-independent split logic...")
splitter = SubjectSplitter(random_seed=42)
unique_subjects = synced_df['subject_id'].unique()
splits = splitter.create_splits(unique_subjects)

os.makedirs('../data/splits/', exist_ok=True)
pd.DataFrame(splits['train']).to_csv('../data/splits/train_ids.csv', index=False, header=['subject_id'])
pd.DataFrame(splits['val']).to_csv('../data/splits/val_ids.csv', index=False, header=['subject_id'])
pd.DataFrame(splits['test']).to_csv('../data/splits/test_ids.csv', index=False, header=['subject_id'])

print("\n==================================================")
print("LOSO SPLIT REPORT")
print("==================================================")
print(f"Train subjects: {splits['train']}")
print(f"Val subjects:   {splits['val']}")
print(f"Test subjects:  {splits['test']}")
print("==================================================")

print("[BACKGROUND] Splitting dataset into clean train, val, and test partitions...")
df_train = synced_df[synced_df['subject_id'].isin(splits['train'])]
df_val = synced_df[synced_df['subject_id'].isin(splits['val'])]
df_test = synced_df[synced_df['subject_id'].isin(splits['test'])]

print("[BACKGROUND] Saving partitioned datasets to data/processed/...")
os.makedirs('../data/processed/train/', exist_ok=True)
os.makedirs('../data/processed/val/', exist_ok=True)
os.makedirs('../data/processed/test/', exist_ok=True)
os.makedirs('../data/processed/metadata/', exist_ok=True)

df_train.to_csv('../data/processed/train/train_synced.csv', index=False)
df_val.to_csv('../data/processed/val/val_synced.csv', index=False)
df_test.to_csv('../data/processed/test/test_synced.csv', index=False)
synced_df.to_csv('../data/processed/metadata/synced_dataset.csv', index=False)
print("[STATUS] Split datasets successfully saved to data/processed/ subfolders.")